# Week 4: Conversational RAG — LangChain LCEL + Memory + Structured Output

This notebook demonstrates the Week 4 pipeline:
- **Retriever**: Local ChromaDB wrapped with `langchain-chroma`
- **LLM**: Gemini 2.5 Flash via `ChatGoogleGenerativeAI` with `.with_structured_output(FinancialSnapshot)`
- **Memory**: `ConversationSummaryBufferMemory` (2,000 token limit, Groq Llama-3 as background summariser)
- **Chain**: LCEL `RunnablePassthrough` pipeline

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from scratch.run_week4 import main
main()

## Inspecting Structured Outputs

After running `main()`, the structured JSON responses are saved to `outputs/responses/week4_responses.json`. Load and inspect them here:

In [ ]:
import json, pathlib

data = json.loads(pathlib.Path('../outputs/responses/week4_responses.json').read_text())
for turn in data:
    print(f"Turn {turn['turn']}: {turn['query']}")
    print(json.dumps(turn['structured_answer'], indent=2))
    print('-' * 60)

## Key Design Decisions

### 1. Why LCEL instead of `create_retrieval_chain`?
`create_retrieval_chain` is a legacy helper from the LangChain 0.3.x line. In LangChain 1.x, the same pattern is expressed with `RunnablePassthrough.assign()` pipes, which are more transparent, testable, and composable.

### 2. Why Gemini as primary + Groq as summariser?
Gemini 2.5 Flash is used as the primary generator — it delivers deep analytical reasoning over the retrieved financial documents and forces structured JSON output via `.with_structured_output()`. Groq's hardware-accelerated Llama-3 handles the background memory summarisation, which is a simpler task that benefits from Groq's ultra-low latency.

### 3. Why `ConversationSummaryBufferMemory`?
Naively appending all messages would eventually fill the model's context window. By summarising older turns, the memory module lets the pipeline hold coherent multi-turn conversations about detailed corporate filings without token overflow.